# Aria Testing Comprehensive Guide

**Complete reference for testing strategies, test patterns, CI/CD validation, and quality gates.**

Learn how to write tests, run test suites, and ensure code quality before deployment.


## Testing Strategy Overview

### Testing Pyramid

```
        △ E2E Tests (5%)
       / \
      /   \  Integration (15%)
     /     \
    /       \ Unit Tests (80%)
   /_________\
```

**Distribution:**

- **Unit Tests (80%):** Fast, isolated, single function
- **Integration Tests (15%):** Module interactions
- **E2E Tests (5%):** Full user workflow

### Test Execution Commands

```bash
# Fast unit tests only
python scripts/test_runner.py --unit
make test-unit

# Run all tests
python scripts/test_runner.py --all
make test

# Integration tests only
python scripts/test_runner.py --integration

# Watch mode (re-run on file change)
python scripts/test_runner.py --unit --watch

# With coverage report
python scripts/test_runner.py --unit --coverage
make test-coverage

# Specific test file
pytest tests/test_function_app.py -v

# Specific test by keyword
pytest -k "chat and not slow" -v

# Show test markers
python scripts/test_runner.py --list-suites
```

---

## Unit Testing Patterns

### Pattern 1: Testing Chat Provider Detection

```python
import pytest
from shared.chat_providers import detect_provider

class TestProviderDetection:

    def test_explicit_provider_override(self, monkeypatch):
        """Explicit provider selection takes precedence"""
        monkeypatch.setenv("DEFAULT_AI_PROVIDER", "azure")
        provider = detect_provider(explicit="local")
        assert provider == "local"

    def test_azure_openai_detection(self, monkeypatch):
        """Azure OpenAI detected when env vars set"""
        monkeypatch.setenv("AZURE_OPENAI_API_KEY", "key123")
        monkeypatch.setenv("AZURE_OPENAI_ENDPOINT", "https://...")
        monkeypatch.setenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4")

        provider = detect_provider()
        assert provider == "azure"

    def test_fallback_to_local(self, monkeypatch):
        """Falls back to local when no providers available"""
        for key in ["AZURE_OPENAI_API_KEY", "OPENAI_API_KEY",
                    "LMSTUDIO_BASE_URL", "OLLAMA_BASE_URL"]:
            monkeypatch.delenv(key, raising=False)

        provider = detect_provider()
        assert provider == "local"
```

### Pattern 2: Testing Async Functions

```python
import pytest

@pytest.mark.asyncio
async def test_async_chat_complete():
    """Test async chat completion"""
    response = await chat_complete_async(
        "Hello",
        model="gpt-3.5-turbo"
    )
    assert len(response) > 0
    assert "response" in response

# Alternative: pytest.mark.asyncio not needed if asyncio_mode = "auto"
async def test_another_async_function():
    """asyncio_mode=auto makes @pytest.mark.asyncio optional"""
    result = await some_async_function()
    assert result is not None
```

### Pattern 3: Testing with Mock

```python
from unittest.mock import Mock, patch
import pytest

def test_chat_with_mock_provider():
    """Test chat behavior when provider returns mock response"""
    with patch('shared.chat_providers.AzureOpenAI') as mock_azure:
        mock_instance = Mock()
        mock_instance.chat.completions.create.return_value.choices[0].message.content = "Mocked response"
        mock_azure.return_value = mock_instance

        response = complete_chat("test", provider="azure")
        assert response == "Mocked response"
        mock_instance.chat.completions.create.assert_called_once()
```

### Pattern 4: Testing Database Operations

```python
import pytest
from shared.sql_engine import SQLEngine

@pytest.fixture
def db():
    """Create in-memory test database"""
    engine = SQLEngine("sqlite:///:memory:")
    yield engine
    engine.close()

def test_save_chat_message(db):
    """Test saving chat message to database"""
    db.save_chat_message(
        session_id="test_123",
        role="user",
        content="Hello"
    )

    messages = db.get_chat_history("test_123")
    assert len(messages) == 1
    assert messages[0]["content"] == "Hello"

def test_connection_pool_saturation(db):
    """Test connection pool behavior under load"""
    # Simulate concurrent connections
    connections = []
    for i in range(db.pool_size + 1):
        try:
            conn = db.get_connection()
            connections.append(conn)
        except Exception as e:
            assert "pool exhausted" in str(e)
            break

    # Clean up
    for conn in connections:
        conn.close()
```

---

## Integration Testing Patterns

### Pattern 1: Testing Aria Command Pipeline

```python
import pytest
from apps.aria.server import parse_command, execute_actions

@pytest.fixture
def aria_client():
    """Create test client for Aria server"""
    from apps.aria.server import app
    return app.test_client()

def test_command_to_actions_pipeline(aria_client):
    """Test full command parsing and execution"""
    response = aria_client.post(
        '/api/aria/command',
        json={'command': 'wave and move left'}
    )

    assert response.status_code == 200
    data = response.get_json()
    assert len(data['actions']) >= 2
    assert any(a['type'] == 'gesture' for a in data['actions'])
    assert any(a['type'] == 'move' for a in data['actions'])

def test_execute_action_sequence(aria_client):
    """Test executing parsed actions"""
    actions = [
        {'type': 'move', 'target': 'center'},
        {'type': 'say', 'text': 'Hello!'},
        {'type': 'gesture', 'gesture': 'wave'}
    ]

    response = aria_client.post(
        '/api/aria/execute',
        json={'actions': actions, 'mode': 'execute'}
    )

    assert response.status_code == 200
    data = response.get_json()
    assert data['status'] == 'completed'
    assert data['actions_executed'] == 3
```

### Pattern 2: Testing API Endpoints

```python
import pytest
from function_app import app

@pytest.fixture
def client():
    """Create test client for Azure Functions"""
    return app.test_client()

def test_chat_endpoint(client):
    """Test /api/chat endpoint"""
    response = client.post(
        '/api/chat',
        json={'message': 'Hello'},
        headers={'Content-Type': 'application/json'}
    )

    assert response.status_code == 200
    data = response.get_json()
    assert 'response' in data
    assert 'provider' in data

def test_status_endpoint_comprehensive(client):
    """Test /api/ai/status returns all expected fields"""
    response = client.get('/api/ai/status')

    assert response.status_code == 200
    data = response.get_json()

    # Check all required fields present
    required_fields = ['provider', 'database', 'azure_openai', 'openai']
    for field in required_fields:
        assert field in data, f"Missing field: {field}"
```

### Pattern 3: Testing Message Processing Pipeline

```python
@pytest.fixture
def chat_session():
    """Set up chat session with memory"""
    from shared.chat_memory import ChatMemory
    return ChatMemory(session_id="test_123")

def test_memory_injection_in_context(chat_session):
    """Test that semantic memory is properly injected"""
    # Store some messages
    chat_session.store_message("user", "Tell me about quantum")
    chat_session.store_message("assistant", "Quantum computing uses qubits")

    # Retrieve similar messages
    similar = chat_session.fetch_similar(
        "What's quantum?",
        k=2
    )

    assert len(similar) >= 1
    assert any("quantum" in m["content"].lower() for m in similar)
```

---

## E2E Testing Patterns

### Pattern 1: Browser-Based E2E Test (Playwright)

```python
import pytest
from playwright.sync_api import sync_playwright

@pytest.fixture
def browser():
    """Set up Playwright browser"""
    with sync_playwright() as p:
        browser = p.chromium.launch()
        yield browser
        browser.close()

def test_aria_ui_full_workflow(browser):
    """Test complete Aria UI interaction flow"""
    page = browser.new_page()
    page.goto("http://localhost:8080")

    # Fill command input
    page.fill('input[type="text"]', "wave and dance")

    # Click execute button
    page.click('button:has-text("Execute")')

    # Wait for animation to complete
    page.wait_for_timeout(2000)

    # Verify Aria moved
    aria = page.query_selector('.aria-character')
    assert aria is not None

    page.close()
```

### Pattern 2: API E2E Test

```python
def test_full_chat_workflow():
    """Test complete chat flow from message to response"""
    # 1. Initialize session
    session_id = "e2e_test_" + datetime.now().isoformat()

    # 2. Send message
    response = requests.post(
        'http://localhost:7071/api/chat',
        json={
            'message': 'What is 2+2?',
            'session_id': session_id
        }
    )
    assert response.status_code == 200

    # 3. Verify response format
    data = response.json()
    assert 'response' in data

    # 4. Check memory was stored
    history = requests.get(
        f'http://localhost:7071/api/chat/history?session_id={session_id}'
    ).json()
    assert len(history) >= 1
```

---

## Test Configuration & Markers

### Pytest Configuration (pytest.ini)

```ini
[pytest]
markers =
    unit: Unit tests (fast, isolated)
    integration: Integration tests (modules interacting)
    e2e: End-to-end tests (full workflows)
    slow: Tests that take >5 seconds
    azure: Tests requiring Azure services
    db: Tests using database
    ai: Tests using AI providers

asyncio_mode = auto
testpaths = tests
python_files = test_*.py
python_classes = Test*
python_functions = test_*
```

### Running Tests by Marker

```bash
# Only fast unit tests
pytest -m "unit and not slow" -v

# Skip Azure tests
pytest -m "not azure" -v

# Only database tests
pytest -m "db" -v

# Exclude slow tests
pytest -m "not slow" -v

# Integration tests only
pytest -m "integration" -v
```

---

## Coverage & Quality

### Generate Coverage Report

```bash
# Create HTML coverage report
python scripts/test_runner.py --unit --coverage
# View: open data_out/coverage_html/index.html

# Terminal coverage summary
pytest tests/ --cov=shared --cov=function_app --cov-report=term-missing

# Fail if coverage < threshold
pytest --cov=shared --cov-fail-under=80
```

### Code Quality Checks

```bash
# Lint
make lint

# Type check
make type-check

# Format check
make lint
# Auto-format
make format

# All quality checks
make lint && make type-check && pytest tests/
```

---

## Continuous Integration

### GitHub Actions Test Workflow

```yaml
name: Tests
on: [push, pull_request]

jobs:
    test:
        runs-on: ubuntu-latest
        strategy:
            matrix:
                python-version: [3.11, 3.12]

        steps:
            - uses: actions/checkout@v3

            - name: Set up Python
              uses: actions/setup-python@v4
              with:
                  python-version: ${{ matrix.python-version }}

            - name: Install dependencies
              run: |
                  pip install -r requirements-dev.txt

            - name: Run unit tests
              run: python scripts/test_runner.py --unit

            - name: Run integration tests
              run: python scripts/test_runner.py --integration

            - name: Upload coverage
              uses: codecov/codecov-action@v3
              with:
                  files: ./coverage.xml
```

---

## Testing Checklist

### Before Committing

- [ ] Unit tests pass: `python scripts/test_runner.py --unit`
- [ ] No lint errors: `make lint`
- [ ] Type checking passes: `make type-check`
- [ ] Coverage > 80%: Check `data_out/coverage_html`

### Before PR

- [ ] Integration tests pass: `python scripts/test_runner.py --integration`
- [ ] E2E tests pass (if applicable): `pytest tests/e2e/`
- [ ] Manual testing of feature
- [ ] Updated tests for new code

### Before Release

- [ ] All tests pass on all Python versions
- [ ] Coverage report reviewed
- [ ] Performance tests pass
- [ ] Security tests pass


## Quick Test Commands

| Command                                | Purpose                | Time     |
| -------------------------------------- | ---------------------- | -------- |
| `pytest tests/`                        | Run all tests          | 5-10 min |
| `python scripts/test_runner.py --unit` | Fast unit tests        | 1-2 min  |
| `pytest tests/ -m "not slow"`          | Exclude slow tests     | 2-3 min  |
| `pytest tests/test_*.py -v`            | Verbose output         | Varies   |
| `pytest -k "chat"`                     | Tests matching keyword | Varies   |
| `pytest --lf`                          | Failed tests only      | <1 min   |
| `make test-coverage`                   | Coverage report        | 5 min    |
| `make lint`                            | Code quality           | <1 min   |

---

## Common Testing Issues

### Issue: Import Error in Tests

```
ModuleNotFoundError: No module named 'shared'
```

**Fix:**

```bash
# Ensure PYTHONPATH includes repo root
export PYTHONPATH=/workspaces/Aria:$PYTHONPATH
pytest tests/
```

### Issue: Async Tests Not Running

```
asyncio_mode not found
```

**Fix:**
Add to `pytest.ini`:

```ini
[pytest]
asyncio_mode = auto
```

### Issue: Database Tests Fail

```
Connection refused: can't connect to database
```

**Fix:**
Use in-memory SQLite:

```python
@pytest.fixture
def db():
    return SQLEngine("sqlite:///:memory:")
```

### Issue: Tests Timeout

```
Timeout: test did not finish within 300s
```

**Fix:**

```bash
# Increase timeout
pytest --timeout=600 tests/

# Or mark test as slow
@pytest.mark.slow
def test_long_running():
    pass
```
